# MUSTAFA MIXING — OCR Credit Scanner
## امسح أي قناة يوتيوب وابحث عن اسم "مصطفى كمال"

⚠️ **الأهم:** اول خطوة، ارفع ملف **cookies.txt** اللي سحبته من Chrome.
   - اذهب إلى Chrome → youtube.com (مسجل دخول)
   - ثبت إضافة "Get cookies.txt LOCALLY"
   - اصدّر الملف
   - ارفعه هنا 👇

In [ ]:
# @title رفع ملف cookies.txt
from google.colab import files
import os

print("📤 ارفع ملف cookies.txt اللي سحبته من Chrome...")
uploaded = files.upload()

if 'cookies.txt' in uploaded:
    print("✅ تم رفع cookies.txt بنجاح!")
else:
    # ممكن اسم الملف مختلف
    for fname in uploaded:
        os.rename(fname, 'cookies.txt')
        print(f"✅ تم إعادة تسمية {fname} إلى cookies.txt")
        break

In [ ]:
# @title تثبيت المكتبات (شغّل مرة واحدة)
!pip install -q yt-dlp easyocr opencv-python pillow torch torchvision
!apt-get -qq install -y tesseract-ocr tesseract-ocr-ara
!pip install -q pytesseract
print("✅ المكتبات جاهزة!")

In [ ]:
# @title مسح قناة يوتيوب
import subprocess
import time
import json
from pathlib import Path
from google.colab import output

# @markdown ---
CHANNEL = "ShababTV" # @param {type:"string"}
MAX_VIDEOS = 20 # @param {type:"integer"}

print(f"🔍 بمسح قناة @{CHANNEL}...")

# Get video list
r = subprocess.run(["yt-dlp", "--cookies", "cookies.txt", "--flat-playlist",
                   "--print", "%(id)s", "--playlist-end", str(MAX_VIDEOS),
                   f"https://www.youtube.com/@{CHANNEL}"],
                  capture_output=True, text=True, timeout=60)
urls = [f"https://www.youtube.com/watch?v={v.strip()}" for v in r.stdout.strip().split('\n') if v.strip()]
print(f"📹 {len(urls)} فيديو تم العثور عليهم")

# OCR setup
import easyocr
import torch
reader = easyocr.Reader(['ar', 'en'], gpu=torch.cuda.is_available())
print(f"🚀 OCR جاهز (GPU: {torch.cuda.is_available()})")

out = Path("ocr_results")
out.mkdir(exist_ok=True)

names = ["مصطفى كمال", "مهندس صوت", "مكس", "ماستر", "مصطفى", "كمال", "Mustafa Kamal"]

import cv2

matches = []
for i, url in enumerate(urls, 1):
    vid = url.split("v=")[-1].split("&")[0]
    print(f"[{i}/{len(urls)}] {vid}...", end=" ")
    
    # Get duration
    r2 = subprocess.run(["yt-dlp", "--cookies", "cookies.txt",
                         "--print", "%(duration)s", "--skip-download", url],
                        capture_output=True, text=True, timeout=15)
    try:
        dur = int(float(r2.stdout.strip()))
    except:
        print("❌ لا يوجد duration")
        continue
    
    if dur < 15:
        print("⏭️ قصير")
        continue
    
    # Download last 10 seconds
    start = max(0, dur - 10)
    mp4 = f"{vid}_outro.mp4"
    r3 = subprocess.run(["yt-dlp", "--cookies", "cookies.txt",
                         "--download-sections", f"*{start}-{dur}",
                         "--force-keyframes-at-cuts",
                         "-f", "worst[ext=mp4]",
                         "-o", mp4, url],
                        capture_output=True, text=True, timeout=40)
    
    if not Path(mp4).exists():
        print("❌ فشل تحميل")
        continue
    
    # Extract frame
    frame = f"{vid}_frame.png"
    subprocess.run(["ffmpeg", "-i", mp4, "-vf", "select=eq(n,0)",
                    "-vsync", "vfr", "-q:v", "2", frame, "-y"],
                   capture_output=True, text=True, timeout=10)
    
    if not Path(frame).exists():
        print("❌ فشل فريم")
        continue
    
    # OCR
    results = reader.readtext(frame)
    texts = [r[1] for r in results]
    print(f"نص: {' | '.join(texts)[:100]}")
    
    for txt in texts:
        for n in names:
            if n.lower() in txt.lower():
                print(f"   ✅✅ وجدت! '{txt}' -> '{n}'")
                matches.append({"url": url, "text": txt, "matched": n})
    
    # Cleanup
    for f in [mp4, frame]:
        p = Path(f)
        if p.exists(): p.unlink()
    
    time.sleep(1)

print(f"\n\n{'='*50}")
print(f"📊 تم مسح {len(urls)} فيديو، {len(matches)} نتيجة")
if matches:
    print("\n✅ النتائج:")
    for m in matches:
        print(f"   🎬 {m['url']}")
        print(f"      '{m['text']}' -> '{m['matched']}'")
else:
    print("\n❌ ما لقينا مطابقات")

with open("scan_report.json", "w", encoding="utf-8") as f:
    json.dump({"matches": matches}, f, ensure_ascii=False, indent=2)
print(f"\n📁 التقرير: scan_report.json")